## Setup libraries and functions

In [ ]:
# Import key librarys
import pandas as pd
import numpy as np

In [ ]:
# Define function to read in excel file
def readfile(year, sheet, rows):
    
        df = pd.read_excel('./school_level/' + year + '.xlsx', 
                                sheet_name=sheet,
                                skiprows=rows,
                                na_values = ['z', 'c','C','x', '#', '*', '.', 0]
                                )
        return df
    
# Define function to read in excel file
# Having 0 as a na_value could lead to problems but many of the older spreadsheets have loads of zero values
def readoldfile(year, sheet, rows):
    
        df = pd.read_excel('./school_level/' + year + '.xlsx',
                                header = [0, 1],
                                sheet_name=sheet,
                                skiprows=rows,
                                na_values = ['z', 'c','C','x', '#', '*', '.', 0]
                                )
        return df
    
# Define tidy up initial df
def tidy(df):
    
    #### Fix usual issues with all strings
    
    # Capitalise all
    df = df.map(lambda x: x.upper() if type(x) is str else x)

    # Strip whitespace
    df = df.map(lambda x: x.strip() if type(x) is str else x)

    # Remove parenthesis
    df = df.map(lambda x: x.replace('(', '') if type(x) is str else x)
    df = df.map(lambda x: x.replace(')', '') if type(x) is str else x)

    # Remove percentage signs
    df = df.map(lambda x: x.replace('%', '') if type(x) is str else x)
    
    # Remove linebreaks
    df = df.map(lambda x: x.replace('\n', '') if type(x) is str else x)

    # Replace annoying substrings
    df = df.map(lambda x: x.replace(' AND ', ' & ') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' – ', ' - ') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' / ', '/') if type(x) is str else x)
    df = df.map(lambda x: x.replace('/ ', '/') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' /', '/') if type(x) is str else x)

    # Strip whitespace again
    df = df.map(lambda x: x.strip() if type(x) is str else x)

    #### Fix usual issues with all columns

    # Capitalise headers
    df.columns = df.columns.astype(str).str.upper()

    # Remove parenthesis from column headers
    df.columns = df.columns.map(lambda x: x.replace('(', '') if type(x) is str else x)
    df.columns = df.columns.map(lambda x: x.replace(')', '') if type(x) is str else x)

    # Remove percentage signs
    df.columns = df.columns.map(lambda x: x.replace('%', '') if type(x) is str else x)
    
    # Remove suffixes from columns
    for i in reversed(range(10)):
        
        df.columns = df.columns.map(lambda x: x.removesuffix(" [NOTE " + str(i) + "]") if type(x) is str else x)
        df.columns = df.columns.map(lambda x: x.removesuffix("[NOTE " + str(i) + "]") if type(x) is str else x)
        df.columns = df.columns.map(lambda x: x.removesuffix(" " + str(i)) if type(x) is str else x)
        df.columns = df.columns.map(lambda x: x.removesuffix(str(i)) if type(x) is str else x)

    # # Remove parenthesis from column headers
    # df.columns = df.columns.map(lambda x: x.replace('(', '') if type(x) is str else x)
    # df.columns = df.columns.map(lambda x: x.replace(')', '') if type(x) is str else x)

    # # Remove suffixes from column headers

    # for i in reversed(range(10)):

    #     df.columns = df.columns.map(lambda x: x.removesuffix("[NOTE " + str(i) + "]") if type(x) is str else x)

    # # Remove remain [Note 8], I think not removed as next to %
    # df.columns = df.columns.str.replace('[NOTE 8]', '', regex=False)
    
    # Strip whitespace from headers
    df.columns = df.columns.map(lambda x: x.strip() if type(x) is str else x)



    return df

# Define tidy up initial df
def tidysimple(df):
    
    #### Fix usual issues with all strings
    
    # Capitalise columns
    df = df.map(lambda x: x.upper() if type(x) is str else x)

    # Strip whitespace
    df = df.map(lambda x: x.strip() if type(x) is str else x)

    # Remove parenthesis
    df = df.map(lambda x: x.replace('(', '') if type(x) is str else x)
    df = df.map(lambda x: x.replace(')', '') if type(x) is str else x)
    
    # Remove linebreaks
    df = df.map(lambda x: x.replace('\n', '') if type(x) is str else x)

    # Replace annoying substrings
    df = df.map(lambda x: x.replace(' AND ', ' & ') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' – ', ' - ') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' / ', '/') if type(x) is str else x)
    df = df.map(lambda x: x.replace('/ ', '/') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' /', '/') if type(x) is str else x)

    return df

## Read in data

In [ ]:
# Create frames to store ldfs
frames = []

In [ ]:
# Read in more recent data

# Create list of years
years = ['2023', '2024', '2025']

# Loop through years
for y in years:
    
    # Read in df
    wdf = readfile(y, y + ' School Level Statistics', 1)

    # Tidy df
    wdf = tidy(wdf)
    
#     # Create new column with SEEDCODE - SCHOOL
#     wdf['ID'] = wdf['SCHOOL NAME'] + ' - ' + wdf['SEEDCODE'].astype(str)
    
    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'LOCAL AUTHORITY': 'LA', 
                                'SCHOOL TYPE': 'TYPE',
                                'PUPIL ROLL': 'ROLL'})

    # Select key columns
    wdf = wdf[['SEEDCODE', 'SCHOOL', 'TYPE', 'LA', 'ROLL']]
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
#ldf.head()

In [ ]:
#wdf.head()

In [ ]:
# Read in old data #1

# Create list of years
years = ['2021', '2022']

# Loop through years
for y in years:
    
    # Read in df
    wdf = readfile(y, y + ' School Level Statistics', 1)

    # Tidy df
    wdf = tidy(wdf)
    
#     # Create new column with SEEDCODE - SCHOOL
#     wdf['ID'] = wdf['SCHOOL NAME'] + ' - ' + wdf['SEEDCODE'].astype(str)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'LOCAL AUTHORITY': 'LA', 
                                'SCHOOL TYPE': 'TYPE',
                                'PUPIL ROLL': 'ROLL'})

    # Select key columns
    wdf = wdf[['SEEDCODE', 'SCHOOL', 'TYPE', 'LA', 'ROLL']]
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
# Read in old data #2

# Create list of years
years = ['2019', '2020']

# Loop through years
for y in years:
    
    # Read in df
    wdf = readfile(y, y, 1)

    # Tidy df
    wdf = tidy(wdf)
    
#     # Create new column with SEEDCODE - SCHOOL
#     wdf['ID'] = wdf['SCHOOL NAME'] + ' - ' + wdf['SEEDCODE'].astype(str)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'LOCAL AUTHORITY': 'LA', 
                                'SCHOOL TYPE': 'TYPE',
                                'PUPIL ROLL': 'ROLL'})

    # Select key columns
    wdf = wdf[['SEEDCODE', 'SCHOOL', 'TYPE', 'LA', 'ROLL']]
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

## Read in older data without attendance figures

In [ ]:
# Read in old data #3

# Create list of years
years = list(reversed(range(2015, 2019)))

# Loop through years
for y in years:

    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 4)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LA NAME': 'LA'})
    
    # Create one column 
    wdf['TYPE'] = wdf['SCHOOL TYPE_PRIMARY'].fillna('') + wdf['SCHOOL TYPE_SECONDARY'].fillna('') \
                    + wdf['SCHOOL TYPE_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
#ldf.head()

In [ ]:
#wdf.columns

In [ ]:
# Read in old data #4

# Create list of years
years = [2014]

# Loop through years
for y in years:
    
    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 6)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LA NAME': 'LA'})
    
    # Create one column 
    wdf['TYPE'] = wdf['SCHOOL TYPE_PRIMARY'].fillna('') + wdf['SCHOOL TYPE_SECONDARY'].fillna('') \
                    + wdf['SCHOOL TYPE_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
# Read in old data #5

# Create list of years
years = list(reversed(range(2011, 2014)))

# Loop through years
for y in years:
    
    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 4)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LA NAME': 'LA'})
    
    # Create one column 
    wdf['TYPE'] = wdf['SCHOOL TYPE_PRIMARY'].fillna('') + wdf['SCHOOL TYPE_SECONDARY'].fillna('') \
                    + wdf['SCHOOL TYPE_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
# Read in old data #5

# Create list of years
years = [2010]

# Loop through years
for y in years:
    
    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 4)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LOCAL AUTHORITY': 'LA',
                                         'CENTRE': 'SEEDCODE'})
    
    # Create one column 
    wdf['TYPE'] = wdf['CONTACT DETAILS_PRIMARY'].fillna('') + wdf['CONTACT DETAILS_SECONDARY'].fillna('') \
                    + wdf['CONTACT DETAILS_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
#wdf.head()

In [ ]:
# Read in old data #6

# Create list of years
years = list(reversed(range(2008, 2010)))

# Loop through years
for y in years:
    
    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 2)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Replace any blank values with NaN
    dfc = dfc.replace(r'^\s*$', np.nan, regex=True)
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LOCAL AUTHORITY': 'LA',
                                         'CENTRE': 'SEEDCODE'})
    
    # Create one column 
    wdf['TYPE'] = wdf['SCHOOL TYPES_PRIMARY'].fillna('') + wdf['SCHOOL TYPES_SECONDARY'].fillna('') \
                    + wdf['SCHOOL TYPES_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
# Read in old data #7

# Create list of years
years = ['2007']

# Loop through years
for y in years:
    
    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 9)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Replace any blank values with NaN
    dfc = dfc.replace(r'^\s*$', np.nan, regex=True)
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LOCAL AUTHORITY': 'LA',
                                         'CENTRE': 'SEEDCODE'})
    
    # Create one column 
    wdf['TYPE'] = wdf['SCHOOL TYPES_PRIMARY'].fillna('') + wdf['SCHOOL TYPES_SECONDARY'].fillna('') \
                    + wdf['SCHOOL TYPES_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
#wdf.head()

In [ ]:
#ldf.head()

In [ ]:
# Read in old data #4

# Create list of years
years = ['2006']

# Loop through years
for y in years:
    
    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 5)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Replace any blank values with NaN
    dfc = dfc.replace(r'^\s*$', np.nan, regex=True)
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LOCAL AUTHORITY': 'LA',
                                         'SEED CODE': 'SEEDCODE'})
    
    # Create one column 
    wdf['TYPE'] = wdf['SCHOOL TYPES_PRIMARY'].fillna('') + wdf['SCHOOL TYPES_SECONDARY'].fillna('') \
                    + wdf['SCHOOL TYPES_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
#wdf.head()

In [ ]:
# Read in old data #4

# Create list of years
years = ['2005']

# Loop through years
for y in years:
    
    # Read in df
    wdf = readoldfile('2018', 'Open at September ' + str(y), 4)
    
    ## Code to sort out multi-index
    # Create df of column headers in multi-index
    dfc = pd.DataFrame(wdf.columns.tolist())
    # Replace any unnamed values with np.nan
    dfc.loc[dfc[0].str.startswith('Unnamed:'), 0] = np.nan
    # Replace any blank values with NaN
    dfc = dfc.replace(r'^\s*$', np.nan, regex=True)
    # Tidy df
    dfc = tidysimple(dfc)
    # Add suffix to first column
    dfc[0] = dfc[0] + '_'
    # Create new column of both values from multi-index
    dfc['HEADERS'] = dfc.fillna('').sum(1).replace('', np.nan)
    # Replace column headers
    wdf.columns = dfc['HEADERS'].tolist()
    
    # Tidy df
    wdf = tidy(wdf)

    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'ADDRESS DETAILS_SCHOOL NAME': 'SCHOOL',
                                    'LOCAL AUTHORITY': 'LA',
                                         'SEED CODE': 'SEEDCODE'})
    
    # Create one column 
    wdf['TYPE'] = wdf['SCHOOL TYPES_PRIMARY'].fillna('') + wdf['SCHOOL TYPES_SECONDARY'].fillna('') \
                    + wdf['SCHOOL TYPES_SPECIAL'].fillna('')
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['SEEDCODE', 'SCHOOL', 'TYPE', 'LA'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='VALUE')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
#wdf.head()

In [ ]:
#ldf.head()

## Join, tidy and export data

In [ ]:
# Concat all ldf together
tdf = pd.concat(frames)

In [ ]:
# Copy total df
tdf_clean = tdf.copy()

# Convert value (pupil roll) column to int
tdf_clean['YEAR'] = tdf_clean['YEAR'].astype(int)

### Use seedcodes to overwrite more recent values over the values from previous years

for y in range(2013, 2026):

    ## School name
    # Select only more recent data
    dfy = tdf_clean.loc[tdf_clean['YEAR'] == y]
    # Create dictionary of seedcode : school pairs
    skey = dict(zip(dfy.SEEDCODE,dfy.SCHOOL))
    # Create new column with 2025 values (fill any blank values with original values)
    tdf_clean['SCHOOL'] = tdf_clean['SEEDCODE'].map(skey).fillna(tdf_clean['SCHOOL'])
    
    ## LA name
    # Select only more recent data
    dfy = tdf_clean.loc[tdf_clean['YEAR'] == y]
    # Create dictionary of seedcode : school pairs
    skey = dict(zip(dfy.SEEDCODE,dfy.LA))
    # Create new column with 2025 values (fill any blank values with original values)
    tdf_clean['LA'] = tdf_clean['SEEDCODE'].map(skey).fillna(tdf_clean['LA'])
    
    ## Type
    # Select only more recent data
    dfy = tdf_clean.loc[tdf_clean['YEAR'] == y]
    # Create dictionary of seedcode : school pairs
    skey = dict(zip(dfy.SEEDCODE,dfy.TYPE))
    # Create new column with 2025 values (fill any blank values with original values)
    tdf_clean['TYPE'] = tdf_clean['SEEDCODE'].map(skey).fillna(tdf_clean['TYPE'])

# Remove any rows with NaN
tdf_clean = tdf_clean.dropna()

# Convert year column to int
tdf_clean['YEAR'] = tdf_clean['YEAR'].astype(int)

# Simplify variable values for pupil rolls
tdf_clean.VARIABLE = tdf_clean.VARIABLE.apply(lambda x: 'PRIMARY ROLL' if 'PRIMARY ROLL' in x else x)
tdf_clean.VARIABLE = tdf_clean.VARIABLE.apply(lambda x: 'PRIMARY ROLL' if 'PUPIL ROLL_PRIMARY' in x else x)
tdf_clean.VARIABLE = tdf_clean.VARIABLE.apply(lambda x: 'PRIMARY ROLL' if 'PUPIL ROLLS_PRIMARY' in x else x)
tdf_clean.VARIABLE = tdf_clean.VARIABLE.apply(lambda x: 'SECONDARY ROLL' if 'SECONDARY ROLL' in x else x)
tdf_clean.VARIABLE = tdf_clean.VARIABLE.apply(lambda x: 'SECONDARY ROLL' if 'PUPIL ROLL_SECONDARY' in x else x)
tdf_clean.VARIABLE = tdf_clean.VARIABLE.apply(lambda x: 'SECONDARY ROLL' if 'PUPIL ROLLS_SECONDARY' in x else x)
tdf_clean.VARIABLE = tdf_clean.VARIABLE.apply(lambda x: 'ROLL' if 'PUPIL ROLL' in x else x)

# Clean up label for joint schools
tdf_clean.TYPE = tdf_clean.TYPE.apply(lambda x: 'PRIMARY&SECONDARY' if 'PRIMARYSECONDARY' in x else x)

# Simplify school names
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' HIGH SCHOOL', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' HIGH', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' ACADEMY', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' GRAMMAR SCHOOL', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' GRAMMAR', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' - SECONDARY SCHOOL', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' - SECONDARY', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' SECONDARY SCHOOL', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' SECONDARY', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' PRIMARY SCHOOL', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' PRIMARY', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' SCHOOL', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' EDUCATION C', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' EDUCATION CENTRE', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' RC', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' R C', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' ASN', '')
# tdf_clean.SCHOOL = tdf_clean.SCHOOL.str.replace(' SPECIAL', '')

# # Rename column
# tdf_clean = tdf_clean.rename(columns={'VALUE': 'ROLL'})

# # Select key columns
# tdf_clean = tdf_clean[['SEEDCODE', 'SCHOOL', 'TYPE', 'LA', 'ROLL', 'YEAR']]

# Select only pupil roll data
tdf_clean = tdf_clean[tdf_clean['VARIABLE'].isin(['ROLL', 'PRIMARY ROLL', 'SECONDARY ROLL'])]

# Convert value (pupil roll) column to int
tdf_clean['VALUE'] = tdf_clean['VALUE'].astype(int)

# Select only schools with pupils in 2025
seeds25 = tdf_clean.loc[tdf_clean['YEAR'] == 2025]['SEEDCODE']
tdf_clean = tdf_clean[tdf_clean['SEEDCODE'].isin(seeds25)]

# Sort values
tdf_clean = tdf_clean.sort_values(by=['YEAR', 'LA', 'TYPE', 'SCHOOL'])

# Reset index
tdf_clean.reset_index(inplace = True, drop = True)

In [ ]:
tdf_clean.to_csv('./csvs/school_level_roll.csv', index=False)

In [ ]:
tdf_clean.head()

In [ ]:
tdf_clean.tail()

In [ ]:
#tdf['VARIABLE'].unique()

In [ ]:
#tdf.tail()

In [ ]:
#testdf = tdf.loc[tdf['YEAR'] == '2005']

In [ ]:
#testdf['VARIABLE'].unique()

In [ ]:
#tdf.head()